In [7]:
from scripts.calculos import Globo
import os
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [8]:


# Diretórios e arquivos
image_dir = 'imagesMasc'
json_path = 'via_export_json.json'

# Lista de imagens
name_images = [image for image in os.listdir(image_dir) if image.endswith('.png')]
image_paths = [os.path.join(image_dir, image_name) for image_name in name_images]

# Dicionário para armazenar imagens
images = {}

# Listas para armazenar médias de cores
milho = []
daninha = []

# Carregar imagens no dicionário
for path, image_name in zip(image_paths, name_images):
    img = cv2.imread(path, cv2.IMREAD_COLOR)  # Carrega em BGR
    img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    images[image_name] = img

# Ler JSON
with open(json_path, 'r') as imagesData:
    datas = json.load(imagesData)

    # Iterar sobre os dados
    for data in [datas[data] for data in datas if datas[data]['regions']]:
        fileName = data['filename']
        
        # Verifica se a imagem foi carregada
        if fileName not in images:
            continue
        
        img = images[fileName]

        # Processar cada região anotada
        for region in data['regions']:
            region_type = region['region_attributes'].get('type', '')

            # Coordenadas do polígono
            all_x = region['shape_attributes']["all_points_x"]
            all_y = region['shape_attributes']["all_points_y"]
            polygon_points = np.array([list(zip(all_x, all_y))], dtype=np.int32)

            # Criar máscara
            mask = np.zeros(img.shape[:2], dtype=np.uint8)
            cv2.fillPoly(mask, polygon_points, 255)

            # Obter os pixels dentro do polígono
            pixels = img[mask == 255]

            # Calcular média de cor (B, G, R)
            if len(pixels) > 0:
                mean_color = tuple(np.mean(pixels, axis=0).astype(int))  # (B, G, R)

                # Armazenar na lista correta
                if region_type == "Milho":
                    milho.append(mean_color)
                elif region_type == "Daninha":
                    daninha.append(mean_color)

# Exibir resultados
print(f"Milho (médias de cor): {milho}")
print(f"Daninha (médias de cor): {daninha}")


Milho (médias de cor): [(np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(141), np.int64(250), np.int64(87)), (np.int64(128), np.int64(245), np.int64(105)), (np.int64(81), np.int64(221), np.int64(174)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(120), np.int64(241), np.int64(117)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(144), np.int64(252), np.int64(84)), (np.int64(136), np.int64(248), np.int64(95)), (np.int64(142), np.int64(250), np.int64(85)), (np.int64(140), np.int64(248), np.int64(89)), (np.int64(27), np.int64(218), np.int64(252)), (np.int64(120), np.int64(241), np.int64(117)), (np.int

In [9]:
milhoGlob = Globo(*milho)
daninhaGlob = Globo(*daninha)
# milhoGlob.exclude(daninhaGlob)
daninhaGlob.exclude(milhoGlob)
# 0 nao e nem minho nem daninha...
# 1 e milho
# 2+ classes exclude de milho

In [10]:
print(milhoGlob.getClass((188,255,140)))
print(milhoGlob.getClass((200,255,140)))
print(milhoGlob.getClass((128,128,128)))



1
0
1


In [11]:
out_path = 'mascs'  # Diretório de saída para as máscaras

# Certifique-se de que a pasta de saída existe
os.makedirs(out_path, exist_ok=True)

# Função para verificar se o índice está dentro dos limites
def in_bounds(y, x, altura, largura):
    return 0 <= y < altura and 0 <= x < largura

# Função que gera os vizinhos de um pixel
def dxdy(x, y, escala):
    for dy in range(-escala, escala + 1):  # Vizinhos em Y
        for dx in range(-escala, escala + 1):  # Vizinhos em X
            if dx == 0 and dy == 0:
                continue  # Ignorar o próprio pixel
            yield y + dy, x + dx

# Percorrer todas as imagens
for image_path, image_name in zip(image_paths, name_images):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)  # Carrega em BGR
    img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)  # Converte para HSV
    
    if img is None:
        print(f"Erro ao carregar a imagem: {image_path}")
        continue

    altura, largura, canais = img.shape

    # Criar uma máscara com fundo 0 (mesmo tamanho da imagem, apenas 1 canal)
    mask = np.zeros((altura, largura), dtype=np.uint8)

    # Definir a escala (tamanho da região ao redor do pixel)
    escala = 5

    # Percorrer pixel a pixel
    for y in range(altura):
        for x in range(largura):
            pixel = tuple(img[y, x])  # (H, S, V) - no caso de HSV
            classe = milhoGlob.getClass(pixel)  # Retorna 0, 1 ou 2
            mask[y, x] = classe
            # # Atualizar a máscara com a classe correspondente para os vizinhos
            # for ny, nx in dxdy(x, y, escala):  # Passa a escala
            #     if in_bounds(ny, nx, altura, largura):  # Se estiver dentro da imagem
            #         mask[ny, nx] = classe

    # Caminho para salvar a máscara
    mask_path = os.path.join(out_path, image_name)

    # Salvar a máscara usando plt
    plt.imsave(mask_path, mask, cmap='gray')

    print(f"Máscara salva em: {mask_path}")


Máscara salva em: mascs\10_image.png
Máscara salva em: mascs\11_image.png


KeyboardInterrupt: 